In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import fbeta_score, confusion_matrix

# LOAD DATA

df = pd.read_csv("data/ai4i2020.csv")

X = df.drop(['UID', 'Product ID', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1)
y = df['Machine failure']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()


# STRATIFIED 5-FOLD (80–20 EACH FOLD)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_test_f2    = []
all_conf_matrices = []
all_depths     = []
all_leaves     = []
all_alphas     = []

best_global_alpha = None
best_global_f2    = -1

# CROSS-VALIDATION LOOP

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

    print(f"\n===== FOLD {fold} =====")

    X_train_full, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train_full, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # FIX 1: Instantiate preprocessor fresh each fold 
    pre = ColumnTransformer([
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

    # FIX 2: Inner train/val split for alpha selection
    # Alpha is chosen on val set → test set is NEVER touched during tuning
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.2,
        stratify=y_train_full,
        random_state=42
    )

    # Fit preprocessing ONLY on inner train split
    X_tr_proc  = pre.fit_transform(X_tr)
    X_val_proc = pre.transform(X_val)
    X_test_proc = pre.transform(X_test)   # same fitted pre, no leakage

    # Pruning path (from inner train only)
    path      = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_tr_proc, y_tr)
    ccp_alphas = path.ccp_alphas

    val_best_f2    = -1
    best_alpha_val = None

    # Select alpha on VALIDATION set
    for alpha in ccp_alphas:
        model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
        model.fit(X_tr_proc, y_tr)
        pred_val = model.predict(X_val_proc)
        f2_val   = fbeta_score(y_val, pred_val, beta=2, zero_division=0)

        # Prefer larger alpha (simpler tree) on ties
        if f2_val > val_best_f2 or (f2_val == val_best_f2 and alpha > best_alpha_val):
            val_best_f2    = f2_val
            best_alpha_val = alpha

    # Retrain on full train fold with chosen alpha
    X_train_proc = pre.fit_transform(X_train_full)   # refit on full train fold
    X_test_proc  = pre.transform(X_test)

    final_model = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha_val)
    final_model.fit(X_train_proc, y_train_full)

    pred_test = final_model.predict(X_test_proc)
    f2_test   = fbeta_score(y_test, pred_test, beta=2, zero_division=0)

    if f2_test == 0:
        print(f"  ⚠️  Warning: F2=0 on fold {fold} — model may predict no positives.")

    cm = confusion_matrix(y_test, pred_test)
    tn, fp, fn, tp = cm.ravel()

    print(f"Best alpha (val)  = {best_alpha_val:.6f}")
    print(f"F2 (val)          = {val_best_f2:.4f}")
    print(f"F2 (test)         = {f2_test:.4f}")
    print(f"Depth = {final_model.get_depth()}, Leaves = {final_model.get_n_leaves()}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    all_test_f2.append(f2_test)
    all_conf_matrices.append(cm)
    all_depths.append(final_model.get_depth())
    all_leaves.append(final_model.get_n_leaves())
    all_alphas.append(best_alpha_val)

    if f2_test > best_global_f2:
        best_global_f2    = f2_test
        best_global_alpha = best_alpha_val

# FINAL SUMMARY

print("\n===== FINAL RESULTS =====")
print(f"Average F2      = {np.mean(all_test_f2):.4f}  ±  {np.std(all_test_f2):.4f}")
print(f"Average Depth   = {np.mean(all_depths):.2f}   ±  {np.std(all_depths):.2f}")
print(f"Average Leaves  = {np.mean(all_leaves):.2f}   ±  {np.std(all_leaves):.2f}")
print(f"Average Alpha   = {np.mean(all_alphas):.6f}  ±  {np.std(all_alphas):.6f}")
print(f"\nBest alpha overall = {best_global_alpha:.6f}  (F2 = {best_global_f2:.4f})")


===== FOLD 1 =====
Best alpha (val)  = 0.000130
F2 (val)          = 0.6589
F2 (test)         = 0.7012
Depth = 17, Leaves = 93
Confusion Matrix: TN=1919, FP=14, FN=21, TP=46

===== FOLD 2 =====
Best alpha (val)  = 0.000103
F2 (val)          = 0.6929
F2 (test)         = 0.6977
Depth = 17, Leaves = 131
Confusion Matrix: TN=1908, FP=24, FN=20, TP=48

===== FOLD 3 =====
Best alpha (val)  = 0.000511
F2 (val)          = 0.5777
F2 (test)         = 0.6748
Depth = 10, Leaves = 22
Confusion Matrix: TN=1922, FP=10, FN=24, TP=44

===== FOLD 4 =====
Best alpha (val)  = 0.000234
F2 (val)          = 0.6346
F2 (test)         = 0.7500
Depth = 10, Leaves = 42
Confusion Matrix: TN=1915, FP=17, FN=17, TP=51

===== FOLD 5 =====
Best alpha (val)  = 0.000264
F2 (val)          = 0.6981
F2 (test)         = 0.5818
Depth = 8, Leaves = 39
Confusion Matrix: TN=1923, FP=9, FN=31, TP=37

===== FINAL RESULTS =====
Average F2      = 0.6811  ±  0.0554
Average Depth   = 12.40   ±  3.83
Average Leaves  = 65.40   ±  40.48